In [ ]:
# ============================================================
# FULL FROM-SCRATCH KAGGLE CODE
# Model: Swin-Base
# ============================================================

!pip install -q timm albumentations opencv-python-headless

import os
import gc
import cv2
import re
import glob
import math
import json
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

class CFG:
    MODEL_TO_RUN = "SwinBase"

    # ========================================================
    # DATA PATHS
    # ========================================================
    ORIGINAL_IMAGE_DIR = "/kaggle/input/datasets/younas12345/msc-research/MS Mining Engineering"
    CSV_PATH = "/kaggle/input/datasets/younas12345/finallabelledcsv/final-labelled-csv_updated.csv"
    TEST_CSV = "/kaggle/input/datasets/younas12345/testgsivaluesupdated/test_gsi.csv"
    TEST_ID_COL = "RM_ID"

    # Working/output paths
    PROCESSED_IMAGE_DIR = "/kaggle/working/processed_gsi_images_swin_base"
    OUTPUT_DIR = "/kaggle/working/gsi_SwinBase_top5_outputs"

    SEED = 42
    IMG_SIZE = 224

    MAX_EPOCHS = 50
    EARLY_STOPPING_PATIENCE = 20
    MIN_DELTA = 1e-4

    TEST_SIZE = 0.20

    # Memory-safe for Swin-Base
    BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 8
    NUM_WORKERS = 0

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_DATAPARALLEL = False

    USE_TARGET_SCALING = True
    USE_TTA = False
    CLIP_PRED_TO_TRAIN_RANGE = True

    TOP_K_CHECKPOINTS = 5

    REUSE_PROCESSED_IF_EXISTS = True
    FORCE_REPROCESS = False

    APPLY_DENOISE_ONCE = False
    APPLY_CLAHE_ONCE = True
    APPLY_SHARPEN_ONCE = True

    WARMUP_EPOCHS = 2
    FREEZE_BACKBONE_EPOCHS = 0

    WEIGHT_DECAY = 1e-4
    LOSS = "mse"

    PLOT_DPI = 900


if os.path.exists(CFG.OUTPUT_DIR):
    shutil.rmtree(CFG.OUTPUT_DIR)

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.PROCESSED_IMAGE_DIR, exist_ok=True)

print("Selected model:", CFG.MODEL_TO_RUN)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("timm:", timm.__version__)
print("Device:", CFG.DEVICE)
print("Batch size:", CFG.BATCH_SIZE)
print("Gradient accumulation:", CFG.GRAD_ACCUM_STEPS)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("WARNING: GPU is not enabled. Use Kaggle Settings > Accelerator > GPU.")


# ============================================================
# 2. SEED AND MEMORY
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


seed_everything(CFG.SEED)
clear_memory()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_dataframe_columns(temp_df):
    temp_df.columns = (
        temp_df.columns.astype(str)
        .str.replace("\t", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return temp_df


def clean_object_columns(temp_df):
    for col in temp_df.columns:
        if temp_df[col].dtype == "object":
            temp_df[col] = (
                temp_df[col].astype(str)
                .str.replace("\t", " ", regex=False)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    temp_df = temp_df.replace(["nan", "NaN", "None", ""], np.nan)

    return temp_df


def make_id_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x


def normalize_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x


def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)


def safe_load_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


# ============================================================
# 4. LOAD AND CLEAN MAIN CSV
# ============================================================

if not os.path.exists(CFG.CSV_PATH):
    raise FileNotFoundError(f"Main CSV not found: {CFG.CSV_PATH}")

df = pd.read_csv(CFG.CSV_PATH)
df = clean_dataframe_columns(df)

df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, regex=True)]
df = clean_object_columns(df)

print("\nCSV shape after cleaning:", df.shape)
print("Cleaned columns:", df.columns.tolist())
display(df.head())

preferred_gsi_cols = ["GSI Value", "GSI", "GSI value", "gsi", "gsi value"]

TARGET_COL = None

for c in preferred_gsi_cols:
    if c in df.columns:
        TARGET_COL = c
        break

if TARGET_COL is None:
    possible_gsi_cols = [c for c in df.columns if "gsi" in c.lower()]

    if len(possible_gsi_cols) == 0:
        raise ValueError("No GSI column found. Rename your target column like 'GSI Value' or 'GSI'.")

    TARGET_COL = possible_gsi_cols[0]

print("Detected target column:", TARGET_COL)

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("Rows after dropping missing GSI:", len(df))


# ============================================================
# 5. PREPROCESS IMAGES ONCE OR REUSE EXISTING PROCESSED IMAGES
# ============================================================

image_exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"]

original_images = []

for ext in image_exts:
    original_images.extend(
        glob.glob(os.path.join(CFG.ORIGINAL_IMAGE_DIR, "**", ext), recursive=True)
    )

if len(original_images) == 0:
    raise FileNotFoundError(f"No original images found. Check ORIGINAL_IMAGE_DIR: {CFG.ORIGINAL_IMAGE_DIR}")

existing_processed = glob.glob(
    os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"),
    recursive=True
)

print("\nOriginal images found:", len(original_images))
print("Existing processed images found:", len(existing_processed))


def preprocess_once(img):
    if CFG.APPLY_DENOISE_ONCE:
        img = cv2.fastNlMeansDenoisingColored(
            img,
            None,
            h=4,
            hColor=4,
            templateWindowSize=7,
            searchWindowSize=21
        )

    if CFG.APPLY_CLAHE_ONCE:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    if CFG.APPLY_SHARPEN_ONCE:
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=1.0)
        img = cv2.addWeighted(img, 1.15, blur, -0.15, 0)

    return img


need_preprocess = True

if (
    CFG.REUSE_PROCESSED_IF_EXISTS
    and len(existing_processed) >= int(0.80 * len(original_images))
    and not CFG.FORCE_REPROCESS
):
    print("Reusing existing processed images. Preprocessing skipped.")
    need_preprocess = False

if need_preprocess:
    print("Preprocessing images once.")

    for img_path in tqdm(original_images):
        rel_path = os.path.relpath(img_path, CFG.ORIGINAL_IMAGE_DIR)
        rel_path_png = str(Path(rel_path).with_suffix(".png"))

        out_path = os.path.join(CFG.PROCESSED_IMAGE_DIR, rel_path_png)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        if os.path.exists(out_path) and not CFG.FORCE_REPROCESS:
            continue

        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = preprocess_once(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        cv2.imwrite(out_path, img)

processed_images = glob.glob(
    os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"),
    recursive=True
)

print("Processed images available:", len(processed_images))

if len(processed_images) == 0:
    raise FileNotFoundError("No processed images available.")

pd.DataFrame({
    "processed_path": processed_images,
    "processed_name": [os.path.basename(p) for p in processed_images],
    "stem": [Path(p).stem for p in processed_images]
}).to_csv(
    os.path.join(CFG.OUTPUT_DIR, "processed_image_manifest.csv"),
    index=False
)

clear_memory()


# ============================================================
# 6. MATCH CSV ROWS TO PROCESSED IMAGES
# ============================================================

by_name = {os.path.basename(p).lower(): p for p in processed_images}
by_stem = {Path(p).stem.lower(): p for p in processed_images}
by_norm_stem = {normalize_key(p): p for p in processed_images}

candidate_image_cols = []

for col in df.columns:
    col_l = col.lower()

    if any(k in col_l for k in ["image", "img", "file", "filename", "path", "rm_id", "id"]):
        candidate_image_cols.append(col)

print("\nCandidate image columns:", candidate_image_cols)


def resolve_image_path(row):
    candidates = []

    for col in candidate_image_cols:
        val = str(row[col]).strip()

        if val == "" or val.lower() == "nan":
            continue

        candidates.append(val)
        candidates.append(os.path.basename(val))
        candidates.append(Path(val).stem)

    for col in df.columns:
        val = str(row[col]).strip()

        if val == "" or val.lower() == "nan":
            continue

        candidates.append(val)
        candidates.append(os.path.basename(val))
        candidates.append(Path(val).stem)

    for cand in candidates:
        cand_l = str(cand).lower().strip()
        cand_norm = normalize_key(cand)

        if cand_l in by_name:
            return by_name[cand_l]

        if cand_l in by_stem:
            return by_stem[cand_l]

        if cand_norm in by_norm_stem:
            return by_norm_stem[cand_norm]

        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
            if cand_l + ext in by_name:
                return by_name[cand_l + ext]

    return None


df["image_path"] = df.apply(resolve_image_path, axis=1)

print("Matched images:", df["image_path"].notna().sum())
print("Missing images:", df["image_path"].isna().sum())

if df["image_path"].isna().sum() > 0:
    print("First missing rows:")
    display(df[df["image_path"].isna()].head(10))

df = df.dropna(subset=["image_path"]).reset_index(drop=True)

if len(df) < 30:
    raise ValueError("Too few matched images. Check CSV identifiers and image filenames.")

print("Final usable samples:", len(df))


# ============================================================
# 7. TRAIN / TEST SPLIT USING GIVEN TEST CSV
# ============================================================

if not os.path.exists(CFG.TEST_CSV):
    raise FileNotFoundError(f"Test CSV not found: {CFG.TEST_CSV}")

test_values = pd.read_csv(CFG.TEST_CSV)
test_values = clean_dataframe_columns(test_values)
test_values = clean_object_columns(test_values)

print("\nTest CSV path:", CFG.TEST_CSV)
print("Test CSV columns:", test_values.columns.tolist())

if CFG.TEST_ID_COL not in test_values.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in test CSV. "
        f"Available columns: {test_values.columns.tolist()}"
    )

if CFG.TEST_ID_COL not in df.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in main CSV. "
        f"Available columns: {df.columns.tolist()}"
    )

df["split_id"] = df[CFG.TEST_ID_COL].apply(make_id_key)
test_values["split_id"] = test_values[CFG.TEST_ID_COL].apply(make_id_key)

test_ids = set(test_values["split_id"].dropna().unique())

test_df = df[df["split_id"].isin(test_ids)].copy()
train_df = df[~df["split_id"].isin(test_ids)].copy()

matched_test_ids = set(test_df["split_id"].unique())
missing_test_ids = sorted(list(test_ids - matched_test_ids))

print("\nTest IDs in given test CSV:", len(test_ids))
print("Matched test samples:", len(test_df))
print("Missing test IDs:", len(missing_test_ids))

if len(missing_test_ids) > 0:
    print("First missing IDs:", missing_test_ids[:20])

if len(test_df) == 0:
    raise ValueError(
        "No test samples matched. Check whether RM_ID values in TEST_CSV match RM_ID values in the main CSV."
    )

if len(train_df) == 0:
    raise ValueError(
        "Training set became empty. Check whether TEST_CSV contains all RM_ID values from the main CSV."
    )

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "train_dataset_used.csv"), index=False)
test_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "test_dataset_used.csv"), index=False)

print("\nTrain samples:", len(train_df))
print("Test samples:", len(test_df))
print("Train GSI range:", train_df[TARGET_COL].min(), "to", train_df[TARGET_COL].max())
print("Test GSI range:", test_df[TARGET_COL].min(), "to", test_df[TARGET_COL].max())


# ============================================================
# 8. TARGET SCALING
# ============================================================

y_mean = float(train_df[TARGET_COL].mean())
y_std = float(train_df[TARGET_COL].std())

if y_std == 0:
    y_std = 1.0

train_min = float(train_df[TARGET_COL].min())
train_max = float(train_df[TARGET_COL].max())

print("\nTarget mean:", y_mean)
print("Target std:", y_std)


def scale_target(y):
    if CFG.USE_TARGET_SCALING:
        return (y - y_mean) / y_std

    return y


def inverse_scale_target(y):
    if CFG.USE_TARGET_SCALING:
        return y * y_std + y_mean

    return y


# ============================================================
# 9. AUGMENTATION
# ============================================================

train_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.20),

    A.Affine(
        scale=(0.94, 1.06),
        translate_percent=(-0.03, 0.03),
        rotate=(-10, 10),
        shear=(-2, 2),
        p=0.50
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.10,
        contrast_limit=0.10,
        p=0.35
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


test_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


# ============================================================
# 10. DATASET AND DATALOADER
# ============================================================

class GSIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(row["image_path"])

        if img is None:
            raise FileNotFoundError(f"Could not read image: {row['image_path']}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        y = torch.tensor(scale_target(float(row[TARGET_COL])), dtype=torch.float32)

        return img, y


train_dataset = GSIDataset(train_df, transform=train_transform)
test_dataset = GSIDataset(test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
    drop_last=False
)

print("\nTrain batches:", len(train_loader))
print("test batches:", len(test_loader))


# ============================================================
# 11. SWIN-BASE MODEL CONFIGURATION
# ============================================================

available_models = timm.list_models()


def pick_model_name(candidates):
    for name in candidates:
        if name in available_models:
            return name

    raise ValueError(f"None of these models are available in this timm version: {candidates}")


model_key = "SwinBase"

config = {
    "candidates": [
        "swin_base_patch4_window7_224.ms_in22k_ft_in1k",
        "swin_base_patch4_window7_224.ms_in1k",
        "swin_base_patch4_window7_224",
        "swin_base_patch4_window12_384.ms_in22k_ft_in1k"
    ],
    "lr": 5e-5,
    "drop_rate": 0.35,
    "drop_path_rate": 0.20
}

config["model_name"] = pick_model_name(config["candidates"])

print("\nSelected model key:", model_key)
print("Selected timm model:", config["model_name"])


# ============================================================
# 12. TRAINING UTILITIES
# ============================================================

def create_model(model_name, drop_rate, drop_path_rate):
    def _try_create(pretrained_flag):
        kwargs = {
            "pretrained": pretrained_flag,
            "num_classes": 1
        }

        if drop_rate is not None:
            kwargs["drop_rate"] = drop_rate

        if drop_path_rate is not None:
            kwargs["drop_path_rate"] = drop_path_rate

        try:
            return timm.create_model(model_name, **kwargs)
        except TypeError:
            kwargs.pop("drop_path_rate", None)

            try:
                return timm.create_model(model_name, **kwargs)
            except TypeError:
                kwargs.pop("drop_rate", None)

                return timm.create_model(model_name, **kwargs)

    try:
        return _try_create(pretrained_flag=True)
    except Exception as e:
        print(f"Pretrained loading failed for {model_name}.")
        print("Reason:", str(e))
        print("Creating model with pretrained=False.")

        return _try_create(pretrained_flag=False)


def unwrap_model(model):
    if isinstance(model, nn.DataParallel):
        return model.module

    return model


def set_backbone_trainable(model, trainable):
    for param in model.parameters():
        param.requires_grad = trainable

    if not trainable:
        for name, param in model.named_parameters():
            name_l = name.lower()

            if any(k in name_l for k in ["head", "classifier", "fc"]):
                param.requires_grad = True


def get_loss_fn():
    if CFG.LOSS == "smoothl1":
        return nn.SmoothL1Loss(beta=0.5)

    if CFG.LOSS == "mse":
        return nn.MSELoss()

    raise ValueError("Unsupported loss function.")


def get_scheduler(optimizer):
    def lr_lambda(epoch):
        if epoch < CFG.WARMUP_EPOCHS:
            return float(epoch + 1) / float(max(1, CFG.WARMUP_EPOCHS))

        progress = float(epoch - CFG.WARMUP_EPOCHS) / float(max(1, CFG.MAX_EPOCHS - CFG.WARMUP_EPOCHS))

        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return True

        improved = score > self.best_score + self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
            return True

        self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True

        return False


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    preds = []
    targets = []

    for images, y in loader:
        images = images.to(CFG.DEVICE, non_blocking=False)
        y = y.to(CFG.DEVICE, non_blocking=False)

        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            out = model(images).view(-1)

        preds.append(out.detach().cpu().numpy())
        targets.append(y.detach().cpu().numpy())

        del images, y, out

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    preds = inverse_scale_target(preds)
    targets = inverse_scale_target(targets)

    if CFG.CLIP_PRED_TO_TRAIN_RANGE:
        preds = np.clip(preds, train_min, train_max)

    clear_memory()

    return targets, preds


def compute_metrics(y_true, y_pred):
    max_len = min(len(y_true), len(y_pred))
    y_true = y_true[:max_len]
    y_pred = y_pred[:max_len]
    
    if len(y_true) < 2:
        return {"R2": 0.0, "RMSE": 0.0, "MSE": 0.0, "MAE": 0.0}
        
    mse = mean_squared_error(y_true, y_pred)

    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred))
    }


def update_top_k_checkpoints(
    top_records,
    model,
    model_key,
    config,
    epoch,
    metrics,
    train_loss,
    test_loss
):
    should_save = False

    if len(top_records) < CFG.TOP_K_CHECKPOINTS:
        should_save = True
    else:
        worst_r2 = min([rec["R2"] for rec in top_records])

        if metrics["R2"] > worst_r2:
            should_save = True

    if not should_save:
        return top_records

    r2_text = f"{metrics['R2']:.5f}".replace("-", "neg").replace(".", "p")

    ckpt_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top_epoch_{epoch:03d}_r2_{r2_text}_state_dict.pth"
    )

    meta_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top_epoch_{epoch:03d}_r2_{r2_text}_metadata.json"
    )

    torch.save(
        unwrap_model(model).state_dict(),
        ckpt_path
    )

    metadata = {
        "model_key": model_key,
        "model_name": config["model_name"],
        "epoch": int(epoch),
        "target_col": TARGET_COL,
        "y_mean": float(y_mean),
        "y_std": float(y_std),
        "train_min": float(train_min),
        "train_max": float(train_max),
        "img_size": int(CFG.IMG_SIZE),
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "metrics": metrics,
        "config": {
            "lr": float(config["lr"]),
            "drop_rate": float(config["drop_rate"]),
            "drop_path_rate": float(config["drop_path_rate"])
        }
    }

    save_json(meta_path, metadata)

    top_records.append({
        "Model": model_key,
        "timm_model_name": config["model_name"],
        "epoch": int(epoch),
        "R2": float(metrics["R2"]),
        "RMSE": float(metrics["RMSE"]),
        "MSE": float(metrics["MSE"]),
        "MAE": float(metrics["MAE"]),
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "state_dict_path": ckpt_path,
        "metadata_path": meta_path
    })

    top_records = sorted(top_records, key=lambda x: x["R2"], reverse=True)

    while len(top_records) > CFG.TOP_K_CHECKPOINTS:
        removed = top_records.pop(-1)

        for p in [removed["state_dict_path"], removed["metadata_path"]]:
            if os.path.exists(p):
                os.remove(p)

    for i, rec in enumerate(top_records, start=1):
        rec["rank"] = i

    pd.DataFrame(top_records).to_csv(
        os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoints.csv"),
        index=False
    )

    return top_records


# ============================================================
# 13. TRAIN SWIN-BASE
# ============================================================

clear_memory()

model = create_model(
    model_name=config["model_name"],
    drop_rate=config["drop_rate"],
    drop_path_rate=config["drop_path_rate"]
)

model = model.to(CFG.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=CFG.WEIGHT_DECAY
)

scheduler = get_scheduler(optimizer)

stopper = EarlyStopping(
    patience=CFG.EARLY_STOPPING_PATIENCE,
    min_delta=CFG.MIN_DELTA
)

loss_fn = get_loss_fn()

use_amp = CFG.DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

history = []
top_checkpoints = []
best_r2 = -999.0

print("\n" + "=" * 90)
print(f"Training selected model only: {model_key}")
print("=" * 90)

for epoch in range(CFG.MAX_EPOCHS):

    if epoch < CFG.FREEZE_BACKBONE_EPOCHS:
        set_backbone_trainable(model, trainable=False)
    else:
        set_backbone_trainable(model, trainable=True)

    model.train()
    train_loss_sum = 0.0

    optimizer.zero_grad(set_to_none=True)

    for step, (images, y) in enumerate(train_loader):
        images = images.to(CFG.DEVICE, non_blocking=False)
        y = y.to(CFG.DEVICE, non_blocking=False)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(images).view(-1)
            loss = loss_fn(out, y)
            loss = loss / CFG.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss_sum += loss.item() * CFG.GRAD_ACCUM_STEPS

        del images, y, out, loss

    train_loss = train_loss_sum / len(train_loader)

    model.eval()
    test_loss_sum = 0.0

    with torch.no_grad():
        for images, y in test_loader:
            images = images.to(CFG.DEVICE, non_blocking=False)
            y = y.to(CFG.DEVICE, non_blocking=False)

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(images).view(-1)
                loss = loss_fn(out, y)

            test_loss_sum += loss.item()

            del images, y, out, loss

    test_loss = test_loss_sum / len(test_loader)

    # Predictions and metrics evaluation
    y_true_train, y_pred_train = predict_model(model, train_loader)
    train_metrics = compute_metrics(y_true_train, y_pred_train)
    
    y_true, y_pred = predict_model(model, test_loader)
    metrics = compute_metrics(y_true, y_pred)

    current_lr = float(optimizer.param_groups[0]["lr"])

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "train_r2": train_metrics["R2"],
        "test_r2": metrics["R2"],
        "test_rmse": metrics["RMSE"],
        "test_mse": metrics["MSE"],
        "test_mae": metrics["MAE"],
        "lr": current_lr
    })

    top_checkpoints = update_top_k_checkpoints(
        top_records=top_checkpoints,
        model=model,
        model_key=model_key,
        config=config,
        epoch=epoch + 1,
        metrics=metrics,
        train_loss=train_loss,
        test_loss=test_loss
    )

    improved = stopper.step(metrics["R2"])

    if improved:
        best_r2 = metrics["R2"]

    scheduler.step()

    print(
        f"Epoch [{epoch+1:02d}/{CFG.MAX_EPOCHS}] | "
        f"LR: {current_lr:.2e} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Test Loss: {test_loss:.5f} | "
        f"Train R2: {train_metrics['R2']:.4f} | "
        f"Test R2: {metrics['R2']:.4f} | "
        f"Top5 Saved: {len(top_checkpoints)} | "
        f"EarlyStop: {stopper.counter}/{CFG.EARLY_STOPPING_PATIENCE}"
    )

    del y_true, y_pred, y_true_train, y_pred_train
    clear_memory()

    if stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch + 1}. Best R2: {best_r2:.4f}")
        break


# ============================================================
# 14. SAVE HISTORY AND TOP-5 CHECKPOINT PREDICTIONS
# ============================================================

history_df = pd.DataFrame(history)

history_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_history.csv")
history_df.to_csv(history_path, index=False)

top_checkpoints = sorted(top_checkpoints, key=lambda x: x["R2"], reverse=True)

top5_index_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoints.csv")
pd.DataFrame(top_checkpoints).to_csv(top5_index_path, index=False)

print("\nTop 5 checkpoints:")
display(pd.DataFrame(top_checkpoints))

checkpoint_eval_rows = []
top5_pred_matrix = []

best_pred_df = None
best_metrics = None
best_path = top_checkpoints[0]["state_dict_path"]

for rank, rec in enumerate(top_checkpoints, start=1):
    clear_memory()

    state_dict = safe_load_state_dict(rec["state_dict_path"], map_location="cpu")
    unwrap_model(model).load_state_dict(state_dict)

    y_true, y_pred = predict_model(model, test_loader)
    ckpt_metrics = compute_metrics(y_true, y_pred)

    pred_df = test_df.copy()
    pred_df["actual_gsi"] = y_true
    pred_df["predicted_gsi"] = y_pred
    pred_df["residual"] = pred_df["actual_gsi"] - pred_df["predicted_gsi"]
    pred_df["absolute_error"] = np.abs(pred_df["residual"])
    pred_df["checkpoint_rank"] = rank
    pred_df["checkpoint_epoch"] = rec["epoch"]
    pred_df["checkpoint_path"] = rec["state_dict_path"]

    pred_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top{rank}_checkpoint_predictions.csv"
    )

    pred_df.to_csv(pred_path, index=False)

    checkpoint_eval_rows.append({
        "Model": model_key,
        "rank": rank,
        "epoch": rec["epoch"],
        "R2": ckpt_metrics["R2"],
        "RMSE": ckpt_metrics["RMSE"],
        "MSE": ckpt_metrics["MSE"],
        "MAE": ckpt_metrics["MAE"],
        "state_dict_path": rec["state_dict_path"],
        "prediction_csv": pred_path
    })

    top5_pred_matrix.append(y_pred)

    if rank == 1:
        best_pred_df = pred_df.copy()
        best_metrics = ckpt_metrics

    del state_dict, y_true, y_pred, pred_df
    clear_memory()

checkpoint_eval_df = pd.DataFrame(checkpoint_eval_rows)

checkpoint_eval_path = os.path.join(
    CFG.OUTPUT_DIR,
    f"{model_key}_top5_checkpoint_evaluation.csv"
)

checkpoint_eval_df.to_csv(checkpoint_eval_path, index=False)

best_prediction_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_best_predictions.csv")
best_pred_df.to_csv(best_prediction_path, index=False)

summary_df = pd.DataFrame([{
    "Model": model_key,
    "timm_model_name": config["model_name"],
    "Best_R2": best_metrics["R2"],
    "Best_RMSE": best_metrics["RMSE"],
    "Best_MSE": best_metrics["MSE"],
    "Best_MAE": best_metrics["MAE"],
    "Best_State_Dict": best_path,
    "Top5_Checkpoint_Index": top5_index_path,
    "Best_Prediction_CSV": best_prediction_path
}])

summary_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\nBest checkpoint:", best_path)
print("Best metrics:", best_metrics)
print("\nSummary:")
display(summary_df)


# ============================================================
# 15. SAVE PREDICTIONS IN ORIGINAL GIVEN TEST CSV ORDER
# ============================================================

given_test_df = pd.read_csv(CFG.TEST_CSV)
given_test_df = clean_dataframe_columns(given_test_df)
given_test_df = clean_object_columns(given_test_df)

if CFG.TEST_ID_COL not in given_test_df.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in given test CSV. "
        f"Available columns: {given_test_df.columns.tolist()}"
    )

given_test_df["split_id"] = given_test_df[CFG.TEST_ID_COL].apply(make_id_key)

best_merge_df = best_pred_df.copy()

if "split_id" not in best_merge_df.columns:
    if CFG.TEST_ID_COL not in best_merge_df.columns:
        raise ValueError(
            f"Column '{CFG.TEST_ID_COL}' not found in best prediction dataframe. "
            f"Available columns: {best_merge_df.columns.tolist()}"
        )

    best_merge_df["split_id"] = best_merge_df[CFG.TEST_ID_COL].apply(make_id_key)

prediction_cols = [
    "split_id",
    "actual_gsi",
    "predicted_gsi",
    "residual",
    "absolute_error",
    "checkpoint_rank",
    "checkpoint_epoch",
    "checkpoint_path"
]

prediction_cols = [c for c in prediction_cols if c in best_merge_df.columns]

given_test_with_predictions = given_test_df.merge(
    best_merge_df[prediction_cols],
    on="split_id",
    how="left"
)

missing_pred_rows = given_test_with_predictions["predicted_gsi"].isna().sum()

given_test_prediction_path = os.path.join(
    CFG.OUTPUT_DIR,
    f"{model_key}_PREDICTIONS_ON_GIVEN_TEST_GSI_CSV.csv"
)

given_test_with_predictions.to_csv(given_test_prediction_path, index=False)

print("\nSaved prediction CSV in original given test CSV order:")
print(given_test_prediction_path)
print("Rows in given test CSV:", len(given_test_df))
print("Rows without prediction after merge:", missing_pred_rows)

if missing_pred_rows > 0:
    missing_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_missing_prediction_rows_from_given_test_csv.csv"
    )

    given_test_with_predictions[
        given_test_with_predictions["predicted_gsi"].isna()
    ].to_csv(missing_path, index=False)

    print("Missing rows saved at:")
    print(missing_path)


# ============================================================
# 16. TOP-5 CHECKPOINT ENSEMBLE FOR SWIN-BASE
# ============================================================

if len(top5_pred_matrix) > 1:
    top5_pred_matrix = np.vstack(top5_pred_matrix)

    ensemble_pred = np.mean(top5_pred_matrix, axis=0)
    y_true_ensemble = best_pred_df["actual_gsi"].values

    ensemble_metrics = compute_metrics(y_true_ensemble, ensemble_pred)

    ensemble_df = test_df.copy()
    ensemble_df["actual_gsi"] = y_true_ensemble
    ensemble_df["predicted_gsi"] = ensemble_pred
    ensemble_df["residual"] = ensemble_df["actual_gsi"] - ensemble_df["predicted_gsi"]
    ensemble_df["absolute_error"] = np.abs(ensemble_df["residual"])

    ensemble_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top5_checkpoint_ensemble_predictions.csv"
    )

    ensemble_df.to_csv(ensemble_path, index=False)

    save_json(
        os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoint_ensemble_metrics.json"),
        ensemble_metrics
    )

    print("\nTop-5 checkpoint ensemble metrics:")
    print(ensemble_metrics)


# ============================================================
# 17. MEMORY-SAFE PLOTS
# ============================================================

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(
        os.path.join(CFG.OUTPUT_DIR, filename),
        dpi=CFG.PLOT_DPI,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()
    clear_memory()


plt.figure(figsize=(8, 5))
plt.hist(train_df[TARGET_COL], bins=12, alpha=0.7, label="Train")
plt.hist(test_df[TARGET_COL], bins=12, alpha=0.7, label="Test")
plt.xlabel("GSI Value")
plt.ylabel("Frequency")
plt.title("GSI Distribution: Train vs Test")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_gsi_distribution_train_test.png")


# --- UPDATED SEPARATE PERFORMANCE PROGRESS PLOTS ---

# 1. Training and Validation Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Training Loss",  linewidth=2)
plt.plot(history_df["epoch"], history_df["test_loss"], label="Validation Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Training and Validation Loss Curve: {model_key}")
plt.legend()
#plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_separate_loss_curve.png")

# 2. Training and Test R2 Curve
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_r2"], label="Train R2",  linewidth=2)
plt.plot(history_df["epoch"], history_df["test_r2"], label="Validation R2", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("R2 Score")
plt.title(f"Train vs Test R2 Progress Curves: {model_key}")
plt.legend()
#plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_separate_r2_curve.png")

# ----------------------------------------------------


y_true_plot = best_pred_df["actual_gsi"].values
y_pred_plot = best_pred_df["predicted_gsi"].values

plt.figure(figsize=(6, 6))
plt.scatter(y_true_plot, y_pred_plot, alpha=0.8)

min_val = min(y_true_plot.min(), y_pred_plot.min())
max_val = max(y_true_plot.max(), y_pred_plot.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual GSI")
plt.ylabel("Predicted GSI")
plt.title(f"Actual vs Predicted GSI: {model_key}")
#plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_actual_vs_predicted.png")


plt.figure(figsize=(7, 5))
plt.hist(best_pred_df["residual"], bins=12, alpha=0.8)
plt.xlabel("Residual: Actual - Predicted")
plt.ylabel("Frequency")
plt.title(f"Residual Distribution: {model_key}")
#plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_residual_distribution.png")


pred_sorted = best_pred_df.sort_values("actual_gsi").reset_index(drop=True)

plt.figure(figsize=(10, 5))
plt.plot(pred_sorted.index, pred_sorted["actual_gsi"], marker="o", label="Actual GSI")
plt.plot(pred_sorted.index, pred_sorted["predicted_gsi"], marker="s", label="Predicted GSI")
plt.xlabel("Test Sample Index Sorted by Actual GSI")
plt.ylabel("GSI Value")
plt.title(f"Actual and Predicted GSI Sequence: {model_key}")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_actual_predicted_sequence.png")


plt.figure(figsize=(10, 5))
plt.plot(pred_sorted.index, pred_sorted["absolute_error"], marker="o")
plt.xlabel("Test Sample Index Sorted by Actual GSI")
plt.ylabel("Absolute Error")
plt.title(f"Absolute Error Per Test Sample: {model_key}")
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_absolute_error_per_sample.png")


# ============================================================
# 18. ZIP OUTPUTS
# ============================================================

del model
clear_memory()

zip_base = CFG.OUTPUT_DIR
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    zip_base,
    "zip",
    CFG.OUTPUT_DIR
)

print("\nAll Swin-Base outputs saved in:", CFG.OUTPUT_DIR)
print("Zipped output saved at:", zip_path)

print("\nImportant files:")
important_files = [
    "train_dataset_used.csv",
    "test_dataset_used.csv",
    f"{model_key}_summary.csv",
    f"{model_key}_top5_checkpoints.csv",
    f"{model_key}_top5_checkpoint_evaluation.csv",
    f"{model_key}_best_predictions.csv",
    f"{model_key}_PREDICTIONS_ON_GIVEN_TEST_GSI_CSV.csv",
    f"{model_key}_top5_checkpoint_ensemble_predictions.csv",
]

for f in important_files:
    p = os.path.join(CFG.OUTPUT_DIR, f)

    if os.path.exists(p):
        print(p)

print("\nGenerated files:")
for f in sorted(os.listdir(CFG.OUTPUT_DIR)):
    print(f)

In [ ]:

# ============================================================
# FULL FROM-SCRATCH KAGGLE CODE
# Model: ConvNeXt-Tiny
#
# Features:
# 1. Complete code in one Kaggle cell.
# 2. Includes image path, updated CSV path, and fixed test CSV path.
# 3. Uses fixed test set from test_gsi.csv using RM_ID.
# 4. Does NOT use predicted_gsi as label.
# 5. Saves top 5 checkpoints based on fixed-test R2.
# 6. Saves predictions in original given test CSV order.
# 7. Saves plots and zipped output.
# 8. Memory-safe settings for ConvNeXt-Tiny.
# ============================================================

!pip install -q timm albumentations opencv-python-headless

import os
import gc
import cv2
import re
import glob
import math
import json
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")



# ============================================================
# 1. CONFIGURATION
# ============================================================

class CFG:
    MODEL_TO_RUN = "ConvNeXtTiny"

    # ========================================================
    # DATA PATHS
    # ========================================================
    ORIGINAL_IMAGE_DIR = "/kaggle/input/datasets/younas12345/msc-research/MS Mining Engineering"
    CSV_PATH = "/kaggle/input/datasets/younas12345/finallabelledcsv/final-labelled-csv_updated.csv"
    TEST_CSV = "/kaggle/input/datasets/younas12345/testgsivaluesupdated/test_gsi.csv"
    TEST_ID_COL = "RM_ID"

    # Working/output paths
    PROCESSED_IMAGE_DIR = "/kaggle/working/processed_gsi_images_convnext_tiny"
    OUTPUT_DIR = "/kaggle/working/gsi_ConvNeXtTiny_top5_outputs"

    SEED = 42
    IMG_SIZE = 224

    MAX_EPOCHS = 50
    EARLY_STOPPING_PATIENCE = 20
    MIN_DELTA = 1e-4

    TEST_SIZE = 0.20

    # Memory-safe for ConvNeXt-Tiny
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 1
    NUM_WORKERS = 0

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_DATAPARALLEL = False

    USE_TARGET_SCALING = True
    USE_TTA = False
    CLIP_PRED_TO_TRAIN_RANGE = True

    TOP_K_CHECKPOINTS = 5

    REUSE_PROCESSED_IF_EXISTS = True
    FORCE_REPROCESS = False

    APPLY_DENOISE_ONCE = False
    APPLY_CLAHE_ONCE = True
    APPLY_SHARPEN_ONCE = True

    WARMUP_EPOCHS = 4
    FREEZE_BACKBONE_EPOCHS = 2

    WEIGHT_DECAY = 1e-4
    LOSS = "smoothl1"

    PLOT_DPI = 900


if os.path.exists(CFG.OUTPUT_DIR):
    shutil.rmtree(CFG.OUTPUT_DIR)

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.PROCESSED_IMAGE_DIR, exist_ok=True)

print("Selected model:", CFG.MODEL_TO_RUN)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("timm:", timm.__version__)
print("Device:", CFG.DEVICE)
print("Batch size:", CFG.BATCH_SIZE)
print("Gradient accumulation:", CFG.GRAD_ACCUM_STEPS)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("WARNING: GPU is not enabled. Use Kaggle Settings > Accelerator > GPU.")



# ============================================================
# 2. SEED AND MEMORY
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


seed_everything(CFG.SEED)
clear_memory()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_dataframe_columns(temp_df):
    temp_df.columns = (
        temp_df.columns.astype(str)
        .str.replace("\t", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return temp_df


def clean_object_columns(temp_df):
    for col in temp_df.columns:
        if temp_df[col].dtype == "object":
            temp_df[col] = (
                temp_df[col].astype(str)
                .str.replace("\t", " ", regex=False)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    temp_df = temp_df.replace(["nan", "NaN", "None", ""], np.nan)

    return temp_df


def make_id_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x


def normalize_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip()

    return x


def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)


def safe_load_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)





#============================================================
# 4. LOAD AND CLEAN MAIN CSV
# ============================================================

if not os.path.exists(CFG.CSV_PATH):
    raise FileNotFoundError(f"Main CSV not found: {CFG.CSV_PATH}")

df = pd.read_csv(CFG.CSV_PATH)
df = clean_dataframe_columns(df)

df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, regex=True)]
df = clean_object_columns(df)

print("\nCSV shape after cleaning:", df.shape)
print("Cleaned columns:", df.columns.tolist())
display(df.head())

preferred_gsi_cols = ["GSI Value", "GSI", "GSI value", "gsi", "gsi value"]

TARGET_COL = None

for c in preferred_gsi_cols:
    if c in df.columns:
        TARGET_COL = c
        break

if TARGET_COL is None:
    possible_gsi_cols = [c for c in df.columns if "gsi" in c.lower()]

    if len(possible_gsi_cols) == 0:
        raise ValueError("No GSI column found. Rename your target column like 'GSI Value' or 'GSI'.")

    TARGET_COL = possible_gsi_cols[0]

print("Detected target column:", TARGET_COL)

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("Rows after dropping missing GSI:", len(df))



# 
# ============================================================
# 5. PREPROCESS IMAGES ONCE OR REUSE EXISTING PROCESSED IMAGES
# ============================================================

image_exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"]

original_images = []

for ext in image_exts:
    original_images.extend(
        glob.glob(os.path.join(CFG.ORIGINAL_IMAGE_DIR, "**", ext), recursive=True)
    )

if len(original_images) == 0:
    raise FileNotFoundError(f"No original images found. Check ORIGINAL_IMAGE_DIR: {CFG.ORIGINAL_IMAGE_DIR}")

existing_processed = glob.glob(
    os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"),
    recursive=True
)

print("\nOriginal images found:", len(original_images))
print("Existing processed images found:", len(existing_processed))


def preprocess_once(img):
    if CFG.APPLY_DENOISE_ONCE:
        img = cv2.fastNlMeansDenoisingColored(
            img,
            None,
            h=4,
            hColor=4,
            templateWindowSize=7,
            searchWindowSize=21
        )

    if CFG.APPLY_CLAHE_ONCE:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    if CFG.APPLY_SHARPEN_ONCE:
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=1.0)
        img = cv2.addWeighted(img, 1.15, blur, -0.15, 0)

    return img


need_preprocess = True

if (
    CFG.REUSE_PROCESSED_IF_EXISTS
    and len(existing_processed) >= int(0.80 * len(original_images))
    and not CFG.FORCE_REPROCESS
):
    print("Reusing existing processed images. Preprocessing skipped.")
    need_preprocess = False

if need_preprocess:
    print("Preprocessing images once.")

    for img_path in tqdm(original_images):
        rel_path = os.path.relpath(img_path, CFG.ORIGINAL_IMAGE_DIR)
        rel_path_png = str(Path(rel_path).with_suffix(".png"))

        out_path = os.path.join(CFG.PROCESSED_IMAGE_DIR, rel_path_png)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        if os.path.exists(out_path) and not CFG.FORCE_REPROCESS:
            continue

        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = preprocess_once(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        cv2.imwrite(out_path, img)

processed_images = glob.glob(
    os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"),
    recursive=True
)

print("Processed images available:", len(processed_images))

if len(processed_images) == 0:
    raise FileNotFoundError("No processed images available.")

pd.DataFrame({
    "processed_path": processed_images,
    "processed_name": [os.path.basename(p) for p in processed_images],
    "stem": [Path(p).stem for p in processed_images]
}).to_csv(
    os.path.join(CFG.OUTPUT_DIR, "processed_image_manifest.csv"),
    index=False
)

clear_memory()


# ============================================================
# 6. MATCH CSV ROWS TO PROCESSED IMAGES
# ============================================================

by_name = {os.path.basename(p).lower(): p for p in processed_images}
by_stem = {Path(p).stem.lower(): p for p in processed_images}
by_norm_stem = {normalize_key(p): p for p in processed_images}

candidate_image_cols = []

for col in df.columns:
    col_l = col.lower()

    if any(k in col_l for k in ["image", "img", "file", "filename", "path", "rm_id", "id"]):
        candidate_image_cols.append(col)

print("\nCandidate image columns:", candidate_image_cols)


def resolve_image_path(row):
    candidates = []

    for col in candidate_image_cols:
        val = str(row[col]).strip()

        if val == "" or val.lower() == "nan":
            continue

        candidates.append(val)
        candidates.append(os.path.basename(val))
        candidates.append(Path(val).stem)

    for col in df.columns:
        val = str(row[col]).strip()

        if val == "" or val.lower() == "nan":
            continue

        candidates.append(val)
        candidates.append(os.path.basename(val))
        candidates.append(Path(val).stem)

    for cand in candidates:
        cand_l = str(cand).lower().strip()
        cand_norm = normalize_key(cand)

        if cand_l in by_name:
            return by_name[cand_l]

        if cand_l in by_stem:
            return by_stem[cand_l]

        if cand_norm in by_norm_stem:
            return by_norm_stem[cand_norm]

        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
            if cand_l + ext in by_name:
                return by_name[cand_l + ext]

    return None


df["image_path"] = df.apply(resolve_image_path, axis=1)

print("Matched images:", df["image_path"].notna().sum())
print("Missing images:", df["image_path"].isna().sum())

if df["image_path"].isna().sum() > 0:
    print("First missing rows:")
    display(df[df["image_path"].isna()].head(10))

df = df.dropna(subset=["image_path"]).reset_index(drop=True)

if len(df) < 30:
    raise ValueError("Too few matched images. Check CSV identifiers and image filenames.")

print("Final usable samples:", len(df))



# ============================================================
# 7. TRAIN / TEST SPLIT USING GIVEN TEST CSV
# Test samples are selected from CFG.TEST_CSV using CFG.TEST_ID_COL.
# ============================================================

if not os.path.exists(CFG.TEST_CSV):
    raise FileNotFoundError(f"Test CSV not found: {CFG.TEST_CSV}")

test_values = pd.read_csv(CFG.TEST_CSV)
test_values = clean_dataframe_columns(test_values)
test_values = clean_object_columns(test_values)

print("\nTest CSV path:", CFG.TEST_CSV)
print("Test CSV columns:", test_values.columns.tolist())

if CFG.TEST_ID_COL not in test_values.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in test CSV. "
        f"Available columns: {test_values.columns.tolist()}"
    )

if CFG.TEST_ID_COL not in df.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in main CSV. "
        f"Available columns: {df.columns.tolist()}"
    )

df["split_id"] = df[CFG.TEST_ID_COL].apply(make_id_key)
test_values["split_id"] = test_values[CFG.TEST_ID_COL].apply(make_id_key)

test_ids = set(test_values["split_id"].dropna().unique())

test_df = df[df["split_id"].isin(test_ids)].copy()
train_df = df[~df["split_id"].isin(test_ids)].copy()

matched_test_ids = set(test_df["split_id"].unique())
missing_test_ids = sorted(list(test_ids - matched_test_ids))

print("\nTest IDs in given test CSV:", len(test_ids))
print("Matched test samples:", len(test_df))
print("Missing test IDs:", len(missing_test_ids))

if len(missing_test_ids) > 0:
    print("First missing IDs:", missing_test_ids[:20])

if len(test_df) == 0:
    raise ValueError(
        "No test samples matched. Check whether RM_ID values in TEST_CSV match RM_ID values in the main CSV."
    )

if len(train_df) == 0:
    raise ValueError(
        "Training set became empty. Check whether TEST_CSV contains all RM_ID values from the main CSV."
    )

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "train_dataset_used.csv"), index=False)
test_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "test_dataset_used.csv"), index=False)

print("\nTrain samples:", len(train_df))
print("Test samples:", len(test_df))
print("Train GSI range:", train_df[TARGET_COL].min(), "to", train_df[TARGET_COL].max())
print("Test GSI range:", test_df[TARGET_COL].min(), "to", test_df[TARGET_COL].max())


# ============================================================
# 8. TARGET SCALING
# ============================================================

y_mean = float(train_df[TARGET_COL].mean())
y_std = float(train_df[TARGET_COL].std())

if y_std == 0:
    y_std = 1.0

train_min = float(train_df[TARGET_COL].min())
train_max = float(train_df[TARGET_COL].max())

print("\nTarget mean:", y_mean)
print("Target std:", y_std)


def scale_target(y):
    if CFG.USE_TARGET_SCALING:
        return (y - y_mean) / y_std

    return y


def inverse_scale_target(y):
    if CFG.USE_TARGET_SCALING:
        return y * y_std + y_mean

    return y


# ============================================================
# 9. AUGMENTATION
# ============================================================

train_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.20),

    A.Affine(
        scale=(0.94, 1.06),
        translate_percent=(-0.03, 0.03),
        rotate=(-10, 10),
        shear=(-2, 2),
        p=0.50
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.10,
        contrast_limit=0.10,
        p=0.35
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


test_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


# ============================================================
# 10. DATASET AND DATALOADER
# ============================================================

class GSIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(row["image_path"])

        if img is None:
            raise FileNotFoundError(f"Could not read image: {row['image_path']}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        y = torch.tensor(scale_target(float(row[TARGET_COL])), dtype=torch.float32)

        return img, y


train_dataset = GSIDataset(train_df, transform=train_transform)
test_dataset = GSIDataset(test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
    drop_last=False
)

print("\nTrain batches:", len(train_loader))
print("test batches:", len(test_loader))


# ============================================================
# 11. CONVNEXT-TINY MODEL CONFIGURATION
# ============================================================

available_models = timm.list_models()


def pick_model_name(candidates):
    for name in candidates:
        if name in available_models:
            return name

    raise ValueError(f"None of these models are available in this timm version: {candidates}")


model_key = "ConvNeXtTiny"

config = {
    "candidates": [
        "convnext_tiny.fb_in22k_ft_in1k",
        "convnext_tiny.fb_in1k",
        "convnext_tiny.in12k_ft_in1k",
        "convnext_tiny"
    ],
    "lr": 1e-4,
    "drop_rate": 0.30,
    "drop_path_rate": 0.10
}

config["model_name"] = pick_model_name(config["candidates"])

print("\nSelected model key:", model_key)
print("Selected timm model:", config["model_name"])


# ============================================================
# 12. TRAINING UTILITIES
# ============================================================

def create_model(model_name, drop_rate, drop_path_rate):
    def _try_create(pretrained_flag):
        kwargs = {
            "pretrained": pretrained_flag,
            "num_classes": 1
        }

        if drop_rate is not None:
            kwargs["drop_rate"] = drop_rate

        if drop_path_rate is not None:
            kwargs["drop_path_rate"] = drop_path_rate

        try:
            return timm.create_model(model_name, **kwargs)
        except TypeError:
            kwargs.pop("drop_path_rate", None)

            try:
                return timm.create_model(model_name, **kwargs)
            except TypeError:
                kwargs.pop("drop_rate", None)

                return timm.create_model(model_name, **kwargs)

    try:
        return _try_create(pretrained_flag=True)
    except Exception as e:
        print(f"Pretrained loading failed for {model_name}.")
        print("Reason:", str(e))
        print("Creating model with pretrained=False.")

        return _try_create(pretrained_flag=False)


def unwrap_model(model):
    if isinstance(model, nn.DataParallel):
        return model.module

    return model


def set_backbone_trainable(model, trainable):
    for param in model.parameters():
        param.requires_grad = trainable

    if not trainable:
        for name, param in model.named_parameters():
            name_l = name.lower()

            if any(k in name_l for k in ["head", "classifier", "fc"]):
                param.requires_grad = True


def get_loss_fn():
    if CFG.LOSS == "smoothl1":
        return nn.SmoothL1Loss(beta=0.5)

    if CFG.LOSS == "mse":
        return nn.MSELoss()

    raise ValueError("Unsupported loss function.")


def get_scheduler(optimizer):
    def lr_lambda(epoch):
        if epoch < CFG.WARMUP_EPOCHS:
            return float(epoch + 1) / float(max(1, CFG.WARMUP_EPOCHS))

        progress = float(epoch - CFG.WARMUP_EPOCHS) / float(max(1, CFG.MAX_EPOCHS - CFG.WARMUP_EPOCHS))

        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return True

        improved = score > self.best_score + self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
            return True

        self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True

        return False


@torch.no_grad()
def predict_model(model, loader):
    model.eval()

    preds = []
    targets = []

    for images, y in loader:
        images = images.to(CFG.DEVICE, non_blocking=False)
        y = y.to(CFG.DEVICE, non_blocking=False)

        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            out = model(images).view(-1)

        preds.append(out.detach().cpu().numpy())
        targets.append(y.detach().cpu().numpy())

        del images, y, out

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    preds = inverse_scale_target(preds)
    targets = inverse_scale_target(targets)

    if CFG.CLIP_PRED_TO_TRAIN_RANGE:
        preds = np.clip(preds, train_min, train_max)

    clear_memory()

    return targets, preds


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mse)),
        "MSE": float(mse),
        "MAE": float(mean_absolute_error(y_true, y_pred))
    }


def update_top_k_checkpoints(
    top_records,
    model,
    model_key,
    config,
    epoch,
    metrics,
    train_loss,
    test_loss
):
    should_save = False

    if len(top_records) < CFG.TOP_K_CHECKPOINTS:
        should_save = True
    else:
        worst_r2 = min([rec["R2"] for rec in top_records])

        if metrics["R2"] > worst_r2:
            should_save = True

    if not should_save:
        return top_records

    r2_text = f"{metrics['R2']:.5f}".replace("-", "neg").replace(".", "p")

    ckpt_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top_epoch_{epoch:03d}_r2_{r2_text}_state_dict.pth"
    )

    meta_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top_epoch_{epoch:03d}_r2_{r2_text}_metadata.json"
    )

    torch.save(
        unwrap_model(model).state_dict(),
        ckpt_path
    )

    metadata = {
        "model_key": model_key,
        "model_name": config["model_name"],
        "epoch": int(epoch),
        "target_col": TARGET_COL,
        "y_mean": float(y_mean),
        "y_std": float(y_std),
        "train_min": float(train_min),
        "train_max": float(train_max),
        "img_size": int(CFG.IMG_SIZE),
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "metrics": metrics,
        "config": {
            "lr": float(config["lr"]),
            "drop_rate": float(config["drop_rate"]),
            "drop_path_rate": float(config["drop_path_rate"])
        }
    }

    save_json(meta_path, metadata)

    top_records.append({
        "Model": model_key,
        "timm_model_name": config["model_name"],
        "epoch": int(epoch),
        "R2": float(metrics["R2"]),
        "RMSE": float(metrics["RMSE"]),
        "MSE": float(metrics["MSE"]),
        "MAE": float(metrics["MAE"]),
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "state_dict_path": ckpt_path,
        "metadata_path": meta_path
    })

    top_records = sorted(top_records, key=lambda x: x["R2"], reverse=True)

    while len(top_records) > CFG.TOP_K_CHECKPOINTS:
        removed = top_records.pop(-1)

        for p in [removed["state_dict_path"], removed["metadata_path"]]:
            if os.path.exists(p):
                os.remove(p)

    for i, rec in enumerate(top_records, start=1):
        rec["rank"] = i

    pd.DataFrame(top_records).to_csv(
        os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoints.csv"),
        index=False
    )

    return top_records


# ============================================================
# 13. TRAIN CONVNEXT-TINY
# ============================================================

clear_memory()

model = create_model(
    model_name=config["model_name"],
    drop_rate=config["drop_rate"],
    drop_path_rate=config["drop_path_rate"]
)

model = model.to(CFG.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=CFG.WEIGHT_DECAY
)

scheduler = get_scheduler(optimizer)

stopper = EarlyStopping(
    patience=CFG.EARLY_STOPPING_PATIENCE,
    min_delta=CFG.MIN_DELTA
)

loss_fn = get_loss_fn()

use_amp = CFG.DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

history = []
top_checkpoints = []
best_r2 = -999.0

print("\n" + "=" * 90)
print(f"Training selected model only: {model_key}")
print("=" * 90)

for epoch in range(CFG.MAX_EPOCHS):

    if epoch < CFG.FREEZE_BACKBONE_EPOCHS:
        set_backbone_trainable(model, trainable=False)
    else:
        set_backbone_trainable(model, trainable=True)

    model.train()
    train_loss_sum = 0.0
    
    # NEW: Initialize tracking arrays for Training Metrics
    train_preds = []
    train_targets = []

    optimizer.zero_grad(set_to_none=True)

    for step, (images, y) in enumerate(train_loader):
        images = images.to(CFG.DEVICE, non_blocking=False)
        y = y.to(CFG.DEVICE, non_blocking=False)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(images).view(-1)
            loss = loss_fn(out, y)
            loss = loss / CFG.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        
        # NEW: Store detached predictions and targets before deletion
        train_preds.append(out.detach().cpu().numpy())
        train_targets.append(y.detach().cpu().numpy())

        if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss_sum += loss.item() * CFG.GRAD_ACCUM_STEPS

        del images, y, out, loss

    train_loss = train_loss_sum / len(train_loader)
    
    # NEW: Compute Training Metrics
    train_preds = np.concatenate(train_preds)
    train_targets = np.concatenate(train_targets)

    train_preds = inverse_scale_target(train_preds)
    train_targets = inverse_scale_target(train_targets)

    if CFG.CLIP_PRED_TO_TRAIN_RANGE:
        train_preds = np.clip(train_preds, train_min, train_max)

    train_metrics = compute_metrics(train_targets, train_preds)

    # Evaluation Phase
    model.eval()
    test_loss_sum = 0.0

    with torch.no_grad():
        for images, y in test_loader:
            images = images.to(CFG.DEVICE, non_blocking=False)
            y = y.to(CFG.DEVICE, non_blocking=False)

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(images).view(-1)
                loss = loss_fn(out, y)

            test_loss_sum += loss.item()

            del images, y, out, loss

    test_loss = test_loss_sum / len(test_loader)

    y_true, y_pred = predict_model(model, test_loader)
    metrics = compute_metrics(y_true, y_pred)

    current_lr = float(optimizer.param_groups[0]["lr"])

    # NEW: Track train_r2 in the history array
    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "test_loss": float(test_loss),
        "train_r2": float(train_metrics["R2"]),
        "test_r2": metrics["R2"],
        "test_rmse": metrics["RMSE"],
        "test_mse": metrics["MSE"],
        "test_mae": metrics["MAE"],
        "lr": current_lr
    })

    top_checkpoints = update_top_k_checkpoints(
        top_records=top_checkpoints,
        model=model,
        model_key=model_key,
        config=config,
        epoch=epoch + 1,
        metrics=metrics,
        train_loss=train_loss,
        test_loss=test_loss
    )

    improved = stopper.step(metrics["R2"])

    if improved:
        best_r2 = metrics["R2"]

    scheduler.step()

    # NEW: Added Train R2 to the print log
    print(
        f"Epoch [{epoch+1:02d}/{CFG.MAX_EPOCHS}] | "
        f"LR: {current_lr:.2e} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Test Loss: {test_loss:.5f} | "
        f"Train R2: {train_metrics['R2']:.4f} | "
        f"Test R2: {metrics['R2']:.4f} | "
        f"RMSE: {metrics['RMSE']:.4f} | "
        f"MSE: {metrics['MSE']:.4f} | "
        f"MAE: {metrics['MAE']:.4f} | "
        f"Top5 Saved: {len(top_checkpoints)} | "
        f"EarlyStop: {stopper.counter}/{CFG.EARLY_STOPPING_PATIENCE}"
    )

    del y_true, y_pred
    clear_memory()

    if stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch + 1}. Best R2: {best_r2:.4f}")
        break

# ============================================================
# 14. SAVE HISTORY AND TOP-5 CHECKPOINT PREDICTIONS
# ============================================================

history_df = pd.DataFrame(history)

history_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_history.csv")
history_df.to_csv(history_path, index=False)

top_checkpoints = sorted(top_checkpoints, key=lambda x: x["R2"], reverse=True)

top5_index_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoints.csv")
pd.DataFrame(top_checkpoints).to_csv(top5_index_path, index=False)

print("\nTop 5 checkpoints:")
display(pd.DataFrame(top_checkpoints))

checkpoint_eval_rows = []
top5_pred_matrix = []

best_pred_df = None
best_metrics = None
best_path = top_checkpoints[0]["state_dict_path"]

for rank, rec in enumerate(top_checkpoints, start=1):
    clear_memory()

    state_dict = safe_load_state_dict(rec["state_dict_path"], map_location="cpu")
    unwrap_model(model).load_state_dict(state_dict)

    y_true, y_pred = predict_model(model, test_loader)
    ckpt_metrics = compute_metrics(y_true, y_pred)

    pred_df = test_df.copy()
    pred_df["actual_gsi"] = y_true
    pred_df["predicted_gsi"] = y_pred
    pred_df["residual"] = pred_df["actual_gsi"] - pred_df["predicted_gsi"]
    pred_df["absolute_error"] = np.abs(pred_df["residual"])
    pred_df["checkpoint_rank"] = rank
    pred_df["checkpoint_epoch"] = rec["epoch"]
    pred_df["checkpoint_path"] = rec["state_dict_path"]

    pred_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top{rank}_checkpoint_predictions.csv"
    )

    pred_df.to_csv(pred_path, index=False)

    checkpoint_eval_rows.append({
        "Model": model_key,
        "rank": rank,
        "epoch": rec["epoch"],
        "R2": ckpt_metrics["R2"],
        "RMSE": ckpt_metrics["RMSE"],
        "MSE": ckpt_metrics["MSE"],
        "MAE": ckpt_metrics["MAE"],
        "state_dict_path": rec["state_dict_path"],
        "prediction_csv": pred_path
    })

    top5_pred_matrix.append(y_pred)

    if rank == 1:
        best_pred_df = pred_df.copy()
        best_metrics = ckpt_metrics

    del state_dict, y_true, y_pred, pred_df
    clear_memory()

checkpoint_eval_df = pd.DataFrame(checkpoint_eval_rows)

checkpoint_eval_path = os.path.join(
    CFG.OUTPUT_DIR,
    f"{model_key}_top5_checkpoint_evaluation.csv"
)

checkpoint_eval_df.to_csv(checkpoint_eval_path, index=False)

best_prediction_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_best_predictions.csv")
best_pred_df.to_csv(best_prediction_path, index=False)

summary_df = pd.DataFrame([{
    "Model": model_key,
    "timm_model_name": config["model_name"],
    "Best_R2": best_metrics["R2"],
    "Best_RMSE": best_metrics["RMSE"],
    "Best_MSE": best_metrics["MSE"],
    "Best_MAE": best_metrics["MAE"],
    "Best_State_Dict": best_path,
    "Top5_Checkpoint_Index": top5_index_path,
    "Best_Prediction_CSV": best_prediction_path
}])

summary_path = os.path.join(CFG.OUTPUT_DIR, f"{model_key}_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\nBest checkpoint:", best_path)
print("Best metrics:", best_metrics)
print("\nSummary:")
display(summary_df)


# ============================================================
# 15. SAVE PREDICTIONS IN ORIGINAL GIVEN TEST CSV ORDER
# ============================================================

given_test_df = pd.read_csv(CFG.TEST_CSV)
given_test_df = clean_dataframe_columns(given_test_df)
given_test_df = clean_object_columns(given_test_df)

if CFG.TEST_ID_COL not in given_test_df.columns:
    raise ValueError(
        f"Column '{CFG.TEST_ID_COL}' not found in given test CSV. "
        f"Available columns: {given_test_df.columns.tolist()}"
    )

given_test_df["split_id"] = given_test_df[CFG.TEST_ID_COL].apply(make_id_key)

best_merge_df = best_pred_df.copy()

if "split_id" not in best_merge_df.columns:
    if CFG.TEST_ID_COL not in best_merge_df.columns:
        raise ValueError(
            f"Column '{CFG.TEST_ID_COL}' not found in best prediction dataframe. "
            f"Available columns: {best_merge_df.columns.tolist()}"
        )

    best_merge_df["split_id"] = best_merge_df[CFG.TEST_ID_COL].apply(make_id_key)

prediction_cols = [
    "split_id",
    "actual_gsi",
    "predicted_gsi",
    "residual",
    "absolute_error",
    "checkpoint_rank",
    "checkpoint_epoch",
    "checkpoint_path"
]

prediction_cols = [c for c in prediction_cols if c in best_merge_df.columns]

given_test_with_predictions = given_test_df.merge(
    best_merge_df[prediction_cols],
    on="split_id",
    how="left"
)

missing_pred_rows = given_test_with_predictions["predicted_gsi"].isna().sum()

given_test_prediction_path = os.path.join(
    CFG.OUTPUT_DIR,
    f"{model_key}_PREDICTIONS_ON_GIVEN_TEST_GSI_CSV.csv"
)

given_test_with_predictions.to_csv(given_test_prediction_path, index=False)

print("\nSaved prediction CSV in original given test CSV order:")
print(given_test_prediction_path)
print("Rows in given test CSV:", len(given_test_df))
print("Rows without prediction after merge:", missing_pred_rows)

if missing_pred_rows > 0:
    missing_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_missing_prediction_rows_from_given_test_csv.csv"
    )

    given_test_with_predictions[
        given_test_with_predictions["predicted_gsi"].isna()
    ].to_csv(missing_path, index=False)

    print("Missing rows saved at:")
    print(missing_path)


# ============================================================
# 16. TOP-5 CHECKPOINT ENSEMBLE FOR CONVNEXT-TINY
# ============================================================

if len(top5_pred_matrix) > 1:
    top5_pred_matrix = np.vstack(top5_pred_matrix)

    ensemble_pred = np.mean(top5_pred_matrix, axis=0)
    y_true_ensemble = best_pred_df["actual_gsi"].values

    ensemble_metrics = compute_metrics(y_true_ensemble, ensemble_pred)

    ensemble_df = test_df.copy()
    ensemble_df["actual_gsi"] = y_true_ensemble
    ensemble_df["predicted_gsi"] = ensemble_pred
    ensemble_df["residual"] = ensemble_df["actual_gsi"] - ensemble_df["predicted_gsi"]
    ensemble_df["absolute_error"] = np.abs(ensemble_df["residual"])

    ensemble_path = os.path.join(
        CFG.OUTPUT_DIR,
        f"{model_key}_top5_checkpoint_ensemble_predictions.csv"
    )

    ensemble_df.to_csv(ensemble_path, index=False)

    save_json(
        os.path.join(CFG.OUTPUT_DIR, f"{model_key}_top5_checkpoint_ensemble_metrics.json"),
        ensemble_metrics
    )

    print("\nTop-5 checkpoint ensemble metrics:")
    print(ensemble_metrics)


# ============================================================
# 17. MEMORY-SAFE PLOTS
# ============================================================

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(
        os.path.join(CFG.OUTPUT_DIR, filename),
        dpi=CFG.PLOT_DPI,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()
    clear_memory()


plt.figure(figsize=(8, 5))
plt.hist(train_df[TARGET_COL], bins=12, alpha=0.7, label="Train")
plt.hist(test_df[TARGET_COL], bins=12, alpha=0.7, label="Test")
plt.xlabel("GSI Value")
plt.ylabel("Frequency")
plt.title("GSI Distribution: Train vs Test")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_gsi_distribution_train_test.png")

# 2. Training and Test R2 Tracking Across Epochs
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_r2"], color="tab:blue", linewidth=2, label="Training R2")
plt.plot(history_df["epoch"], history_df["test_r2"],  color="tab:green", linewidth=2, label="Validation R2")
plt.xlabel("Epoch")
plt.ylabel("R2 Score")
#plt.title(f"Training vs Test R2 History: {model_key}")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_r2_curve12.png")


plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["test_r2"])
plt.xlabel("Epoch")
plt.ylabel("Test R2")
plt.title(f"R2 Progress Over Epochs: {model_key}")
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_r2_curve.png")


plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Training Loss")
plt.plot(history_df["epoch"], history_df["test_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Training and Test Loss: {model_key}")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_loss_curve.png")


y_true_plot = best_pred_df["actual_gsi"].values
y_pred_plot = best_pred_df["predicted_gsi"].values

plt.figure(figsize=(6, 6))
plt.scatter(y_true_plot, y_pred_plot, alpha=0.8)

min_val = min(y_true_plot.min(), y_pred_plot.min())
max_val = max(y_true_plot.max(), y_pred_plot.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual GSI")
plt.ylabel("Predicted GSI")
plt.title(f"Actual vs Predicted GSI: {model_key}")
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_actual_vs_predicted.png")


plt.figure(figsize=(7, 5))
plt.hist(best_pred_df["residual"], bins=12, alpha=0.8)
plt.xlabel("Residual: Actual - Predicted")
plt.ylabel("Frequency")
plt.title(f"Residual Distribution: {model_key}")
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_residual_distribution.png")


pred_sorted = best_pred_df.sort_values("actual_gsi").reset_index(drop=True)

plt.figure(figsize=(10, 5))
plt.plot(pred_sorted.index, pred_sorted["actual_gsi"], marker="o", label="Actual GSI")
plt.plot(pred_sorted.index, pred_sorted["predicted_gsi"], marker="s", label="Predicted GSI")
plt.xlabel("Test Sample Index Sorted by Actual GSI")
plt.ylabel("GSI Value")
plt.title(f"Actual and Predicted GSI Sequence: {model_key}")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_actual_predicted_sequence.png")


plt.figure(figsize=(10, 5))
plt.plot(pred_sorted.index, pred_sorted["absolute_error"], marker="o")
plt.xlabel("Test Sample Index Sorted by Actual GSI")
plt.ylabel("Absolute Error")
plt.title(f"Absolute Error Per Test Sample: {model_key}")
plt.grid(True, alpha=0.3)
save_plot(f"{model_key}_absolute_error_per_sample.png")


# ============================================================
# 18. ZIP OUTPUTS
# ============================================================

del model
clear_memory()

zip_base = CFG.OUTPUT_DIR
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    zip_base,
    "zip",
    CFG.OUTPUT_DIR
)

print("\nAll ConvNeXt-Tiny outputs saved in:", CFG.OUTPUT_DIR)
print("Zipped output saved at:", zip_path)

print("\nImportant files:")
important_files = [
    "train_dataset_used.csv",
    "test_dataset_used.csv",
    f"{model_key}_summary.csv",
    f"{model_key}_top5_checkpoints.csv",
    f"{model_key}_top5_checkpoint_evaluation.csv",
    f"{model_key}_best_predictions.csv",
    f"{model_key}_PREDICTIONS_ON_GIVEN_TEST_GSI_CSV.csv",
    f"{model_key}_top5_checkpoint_ensemble_predictions.csv",
]

for f in important_files:
    p = os.path.join(CFG.OUTPUT_DIR, f)

    if os.path.exists(p):
        print(p)

print("\nGenerated files:")
for f in sorted(os.listdir(CFG.OUTPUT_DIR)):
    print(f)

In [ ]:
# ============================================================
# FULL FROM-SCRATCH KAGGLE CODE (PUBLICATION-QUALITY VISUALS)
# Model: CustomRockCNN | Strategy: 5-Fold Cross-Validation
# ============================================================

!pip install -q albumentations opencv-python-headless

import os
import gc
import cv2
import re
import glob
import math
import json
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

class CFG:
    MODEL_TO_RUN = "CustomRockCNN"

    # ========================================================
    # DATA PATHS
    # ========================================================
    ORIGINAL_IMAGE_DIR = "/kaggle/input/datasets/younas12345/msc-research/MS Mining Engineering"
    CSV_PATH = "/kaggle/input/datasets/younas12345/finallabelledcsv/final-labelled-csv_updated.csv"
    TEST_CSV = "/kaggle/input/datasets/younas12345/testgsivaluesupdated/test_gsi.csv"
    TEST_ID_COL = "RM_ID"

    WORKING_DIR = "/kaggle/working"
    PROCESSED_IMAGE_DIR = "/kaggle/working/processed_gsi_images_custom_cnn"
    OUTPUT_DIR = "/kaggle/working/gsi_CustomCNN_kfold_outputs"

    SEED = 42
    IMG_SIZE = 224

    MAX_EPOCHS = 60             
    EARLY_STOPPING_PATIENCE = 10
    MIN_DELTA = 1e-4
    
    N_FOLDS = 5

    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 2
    NUM_WORKERS = 2

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_TARGET_SCALING = True
    CLIP_PRED_TO_TRAIN_RANGE = True

    REUSE_PROCESSED_IF_EXISTS = True
    FORCE_REPROCESS = False

    APPLY_DENOISE_ONCE = False
    APPLY_CLAHE_ONCE = True
    APPLY_SHARPEN_ONCE = True

    WARMUP_EPOCHS = 4
    WEIGHT_DECAY = 1e-4
    LOSS = "smoothl1"
    PLOT_DPI = 300  # High resolution for publication-ready figures


if os.path.exists(CFG.OUTPUT_DIR):
    shutil.rmtree(CFG.OUTPUT_DIR)

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.PROCESSED_IMAGE_DIR, exist_ok=True)

print("Selected Model:", CFG.MODEL_TO_RUN)
print("Cross-Validation Folds:", CFG.N_FOLDS)
print("Compute Device Engine:", CFG.DEVICE)


# ============================================================
# 2. SEED AND MEMORY MANAGEMENT
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

seed_everything(CFG.SEED)
clear_memory()


# ============================================================
# 3. UTILITY METHODS
# ============================================================

def clean_dataframe_columns(temp_df):
    temp_df.columns = (
        temp_df.columns.astype(str)
        .str.replace("\t", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return temp_df

def clean_object_columns(temp_df):
    for col in temp_df.columns:
        if temp_df[col].dtype == "object":
            temp_df[col] = (
                temp_df[col].astype(str)
                .str.replace("\t", " ", regex=False)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )
    temp_df = temp_df.replace(["nan", "NaN", "None", ""], np.nan)
    return temp_df

def make_id_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def normalize_key(x):
    x = str(x)
    x = os.path.basename(x)
    x = Path(x).stem
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)

def safe_load_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


# ============================================================
# 4. DATA EXTRACTION AND TARGET CLEANING
# ============================================================

if not os.path.exists(CFG.CSV_PATH):
    raise FileNotFoundError(f"Main data matrix file missing: {CFG.CSV_PATH}")

df = pd.read_csv(CFG.CSV_PATH)
df = clean_dataframe_columns(df)
df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, regex=True)]
df = clean_object_columns(df)

preferred_gsi_cols = ["GSI Value", "GSI", "GSI value", "gsi", "gsi value"]
TARGET_COL = None

for c in preferred_gsi_cols:
    if c in df.columns:
        TARGET_COL = c
        break

if TARGET_COL is None:
    possible_gsi_cols = [c for c in df.columns if "gsi" in c.lower()]
    if len(possible_gsi_cols) == 0:
        raise ValueError("Target regression tracking variable 'GSI' not located.")
    TARGET_COL = possible_gsi_cols[0]

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)


# ============================================================
# 5. IMAGE OPTIMIZATION SUBROUTINES
# ============================================================

image_exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"]
original_images = []
for ext in image_exts:
    original_images.extend(glob.glob(os.path.join(CFG.ORIGINAL_IMAGE_DIR, "**", ext), recursive=True))

existing_processed = glob.glob(os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"), recursive=True)

def preprocess_once(img):
    if CFG.APPLY_DENOISE_ONCE:
        img = cv2.fastNlMeansDenoisingColored(img, None, h=4, hColor=4, templateWindowSize=7, searchWindowSize=21)
    if CFG.APPLY_CLAHE_ONCE:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    if CFG.APPLY_SHARPEN_ONCE:
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=1.0)
        img = cv2.addWeighted(img, 1.15, blur, -0.15, 0)
    return img

need_preprocess = True
if CFG.REUSE_PROCESSED_IF_EXISTS and len(existing_processed) >= int(0.80 * len(original_images)) and not CFG.FORCE_REPROCESS:
    need_preprocess = False

if need_preprocess:
    for img_path in tqdm(original_images, desc="Running Preprocessing Filters"):
        rel_path = os.path.relpath(img_path, CFG.ORIGINAL_IMAGE_DIR)
        rel_path_png = str(Path(rel_path).with_suffix(".png"))
        out_path = os.path.join(CFG.PROCESSED_IMAGE_DIR, rel_path_png)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        if os.path.exists(out_path) and not CFG.FORCE_REPROCESS:
            continue
        img = cv2.imread(img_path)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = preprocess_once(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(out_path, img)

processed_images = glob.glob(os.path.join(CFG.PROCESSED_IMAGE_DIR, "**", "*.png"), recursive=True)


# ============================================================
# 6. FILE KEY PAIRING
# ============================================================

by_name = {os.path.basename(p).lower(): p for p in processed_images}
by_stem = {Path(p).stem.lower(): p for p in processed_images}
by_norm_stem = {normalize_key(p): p for p in processed_images}

candidate_image_cols = [c for c in df.columns if any(k in c.lower() for k in ["image", "img", "file", "filename", "path", "rm_id", "id"])]

def resolve_image_path(row):
    candidates = []
    for col in candidate_image_cols:
        val = str(row[col]).strip()
        if val == "" or val.lower() == "nan": continue
        candidates.extend([val, os.path.basename(val), Path(val).stem])
    for col in df.columns:
        val = str(row[col]).strip()
        if val == "" or val.lower() == "nan": continue
        candidates.extend([val, os.path.basename(val), Path(val).stem])
    for cand in candidates:
        cand_l = str(cand).lower().strip()
        cand_norm = normalize_key(cand)
        if cand_l in by_name: return by_name[cand_l]
        if cand_l in by_stem: return by_stem[cand_l]
        if cand_norm in by_norm_stem: return by_norm_stem[cand_norm]
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
            if cand_l + ext in by_name: return by_name[cand_l + ext]
    return None

df["image_path"] = df.apply(resolve_image_path, axis=1)
df = df.dropna(subset=["image_path"]).reset_index(drop=True)


# ============================================================
# 7. HOLDOUT TEST ISOLATION & K-FOLD DEPLOYMENT
# ============================================================

if not os.path.exists(CFG.TEST_CSV):
    raise FileNotFoundError(f"Holdout confirmation file unavailable: {CFG.TEST_CSV}")

test_values = pd.read_csv(CFG.TEST_CSV)
test_values = clean_dataframe_columns(test_values)
test_values = clean_object_columns(test_values)

df["split_id"] = df[CFG.TEST_ID_COL].apply(make_id_key)
test_values["split_id"] = test_values[CFG.TEST_ID_COL].apply(make_id_key)
test_ids = set(test_values["split_id"].dropna().unique())

test_df = df[df["split_id"].isin(test_ids)].copy().reset_index(drop=True)
train_pool_df = df[~df["split_id"].isin(test_ids)].copy().reset_index(drop=True)

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
train_pool_df["fold"] = -1
for fold, (train_idx, val_idx) in enumerate(kf.split(train_pool_df)):
    train_pool_df.loc[val_idx, "fold"] = fold

test_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "test_dataset_used.csv"), index=False)
train_pool_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "kfold_training_pool.csv"), index=False)

print(f"\nFinal Split Matrix Design:\nTraining Development Pool: {len(train_pool_df)} rows distributed across {CFG.N_FOLDS} folds.")
print(f"Isolated Validation Size Per Fold: ~{len(train_pool_df)//CFG.N_FOLDS} rows | Static Test Holdout Size: {len(test_df)}")


# ============================================================
# 8. GEOMETRIC & COLOR DATA AUGMENTATIONS
# ============================================================

train_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.12, rotate_limit=20, p=0.6, border_mode=cv2.BORDER_CONSTANT),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

eval_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


# ============================================================
# 9. PYTORCH DATA SOURCE HANDLERS
# ============================================================

class GSIDataset(Dataset):
    def __init__(self, dataframe, transform=None, y_mean=0.0, y_std=1.0):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.y_mean = y_mean
        self.y_std = y_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["image_path"])
        if img is None: raise FileNotFoundError(f"Unreadable image track: {row['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        
        target_val = float(row[TARGET_COL])
        if CFG.USE_TARGET_SCALING:
            target_val = (target_val - self.y_mean) / self.y_std
            
        y = torch.tensor(target_val, dtype=torch.float32)
        return img, y


# ============================================================
# 10. CUSTOM CNN MODEL ARCHITECTURE DEPLOYMENT
# ============================================================

class CustomRockCNN(nn.Module):
    def __init__(self, drop_rate=0.30):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.10)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15)
        )
        
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.20)
        )
        
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        self.regressor = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(drop_rate),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x).flatten(1)
        x = self.regressor(x)
        return x


# ============================================================
# 11. RECURRENT TRAINING SUBSTRUCTURE
# ============================================================

def get_loss_fn():
    return nn.SmoothL1Loss(beta=0.5) if CFG.LOSS == "smoothl1" else nn.MSELoss()

def get_scheduler(optimizer):
    def lr_lambda(epoch):
        if epoch < CFG.WARMUP_EPOCHS: return float(epoch + 1) / float(max(1, CFG.WARMUP_EPOCHS))
        progress = float(epoch - CFG.WARMUP_EPOCHS) / float(max(1, CFG.MAX_EPOCHS - CFG.WARMUP_EPOCHS))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_score = None; self.counter = 0; self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return True
        if score > self.best_score + self.min_delta:
            self.best_score = score; self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience: self.should_stop = True
        return False

@torch.no_grad()
def predict_model(model, loader, y_mean, y_std, train_min, train_max):
    model.eval()
    preds, targets = [], []
    for images, y in loader:
        images = images.to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            out = model(images).view(-1)
        preds.append(out.detach().cpu().numpy())
        targets.append(y.numpy())
    preds = np.concatenate(preds); targets = np.concatenate(targets)
    
    if CFG.USE_TARGET_SCALING:
        preds = preds * y_std + y_mean
        targets = targets * y_std + y_mean
        
    if CFG.CLIP_PRED_TO_TRAIN_RANGE: 
        preds = np.clip(preds, train_min, train_max)
    clear_memory()
    return targets, preds

def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"R2": float(r2_score(y_true, y_pred)), "RMSE": float(np.sqrt(mse)), "MSE": float(mse), "MAE": float(mean_absolute_error(y_true, y_pred))}


# ============================================================
# 12. RUN ENTIRE 5-FOLD CROSS-VALIDATION LOOP
# ============================================================

oof_predictions = np.zeros(len(train_pool_df))
test_predictions_matrix = np.zeros((CFG.N_FOLDS, len(test_df)))
fold_histories = {}

global_train_min = float(train_pool_df[TARGET_COL].min())
global_train_max = float(train_pool_df[TARGET_COL].max())

print(f"\nInitiating Custom CNN Cross-Validation Pipeline ({CFG.N_FOLDS} Folds)...")

for fold in range(CFG.N_FOLDS):
    print(f"\n" + "="*50 + f"\nTRAINING CROSS-VALIDATION FOLD {fold+1}/{CFG.N_FOLDS}\n" + "="*50)
    
    fold_train_df = train_pool_df[train_pool_df["fold"] != fold].reset_index(drop=True)
    fold_val_df = train_pool_df[train_pool_df["fold"] == fold].reset_index(drop=True)
    
    f_mean = float(fold_train_df[TARGET_COL].mean())
    f_std = float(fold_train_df[TARGET_COL].std()) if float(fold_train_df[TARGET_COL].std()) != 0 else 1.0
    f_min = float(fold_train_df[TARGET_COL].min())
    f_max = float(fold_train_df[TARGET_COL].max())
    
    fold_train_ds = GSIDataset(fold_train_df, transform=train_transform, y_mean=f_mean, y_std=f_std)
    fold_val_ds = GSIDataset(fold_val_df, transform=eval_transform, y_mean=f_mean, y_std=f_std)
    fold_test_ds = GSIDataset(test_df, transform=eval_transform, y_mean=f_mean, y_std=f_std)
    
    fold_train_loader = DataLoader(fold_train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, drop_last=False)
    fold_val_loader = DataLoader(fold_val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, drop_last=False)
    fold_test_loader = DataLoader(fold_test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, drop_last=False)
    
    clear_memory()
    model = CustomRockCNN(drop_rate=0.30).to(CFG.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer)
    stopper = EarlyStopping(patience=CFG.EARLY_STOPPING_PATIENCE, min_delta=CFG.MIN_DELTA)
    loss_fn = get_loss_fn()
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))
    
    best_fold_r2 = -999.0
    fold_history = []
    fold_ckpt_path = os.path.join(CFG.OUTPUT_DIR, f"{CFG.MODEL_TO_RUN}_fold_{fold}_best_model.pth")
    
    for epoch in range(CFG.MAX_EPOCHS):
        model.train()
        train_loss_sum = 0.0
        optimizer.zero_grad(set_to_none=True)
        
        for step, (images, y) in enumerate(fold_train_loader):
            images, y = images.to(CFG.DEVICE), y.to(CFG.DEVICE)
            with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
                out = model(images).view(-1)
                loss = loss_fn(out, y) / CFG.GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            
            if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(fold_train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            train_loss_sum += loss.item() * CFG.GRAD_ACCUM_STEPS
            
        train_loss = train_loss_sum / len(fold_train_loader)
        
        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for images, y in fold_val_loader:
                images, y = images.to(CFG.DEVICE), y.to(CFG.DEVICE)
                with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
                    out = model(images).view(-1)
                    loss = loss_fn(out, y)
                val_loss_sum += loss.item()
                
        val_loss = val_loss_sum / len(fold_val_loader)
        val_true, val_pred = predict_model(model, fold_val_loader, f_mean, f_std, f_min, f_max)
        val_metrics = compute_metrics(val_true, val_pred)
        
        fold_history.append({
            "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, "val_r2": val_metrics["R2"]
        })
        
        if val_metrics["R2"] > best_fold_r2:
            best_fold_r2 = val_metrics["R2"]
            torch.save(model.state_dict(), fold_ckpt_path)
            
        improved = stopper.step(val_metrics["R2"])
        scheduler.step()
        
        print(f"Epoch [{epoch+1:02d}/{CFG.MAX_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val R2: {val_metrics['R2']:.4f}")
        if stopper.should_stop:
            print("Early convergence threshold triggered.")
            break
            
    fold_histories[fold] = pd.DataFrame(fold_history)
    
    clear_memory()
    best_state = safe_load_state_dict(fold_ckpt_path, map_location="cpu")
    model.load_state_dict(best_state)
    
    _, val_oof_preds = predict_model(model, fold_val_loader, f_mean, f_std, f_min, f_max)
    val_indices = train_pool_df[train_pool_df["fold"] == fold].index
    oof_predictions[val_indices] = val_oof_preds
    
    _, test_fold_preds = predict_model(model, fold_test_loader, f_mean, f_std, f_min, f_max)
    test_predictions_matrix[fold, :] = test_fold_preds
    
    del model, optimizer, scheduler, fold_train_loader, fold_val_loader, fold_test_loader
    clear_memory()


# ============================================================
# 13. CROSS-VALIDATION ANALYSIS & OOF EXPORTATION
# ============================================================

print("\n" + "="*50 + "\nCROSS-VALIDATION DEVELOPMENT PERFORMANCE PROFILE\n" + "="*50)
y_true_oof = train_pool_df[TARGET_COL].values
oof_metrics = compute_metrics(y_true_oof, oof_predictions)

print(f"Overall OOF Validation R2:  {oof_metrics['R2']:.5f}")
print(f"Overall OOF Validation RMSE: {oof_metrics['RMSE']:.5f}")
print(f"Overall OOF Validation MAE:  {oof_metrics['MAE']:.5f}")

oof_export_df = train_pool_df.copy()
oof_export_df["oof_predicted_gsi"] = oof_predictions
oof_export_df["oof_residual"] = oof_export_df[TARGET_COL] - oof_export_df["oof_predicted_gsi"]
oof_export_df.to_csv(os.path.join(CFG.OUTPUT_DIR, f"{CFG.MODEL_TO_RUN}_OOF_VALIDATION_PREDICTIONS.csv"), index=False)


# ============================================================
# 14. 5-FOLD K-FOLD ENSEMBLE TESTING
# ============================================================

ensemble_test_preds = np.mean(test_predictions_matrix, axis=0)
y_true_test = test_df[TARGET_COL].values
ensemble_test_metrics = compute_metrics(y_true_test, ensemble_test_preds)

print("\n" + "="*50 + "\n5-FOLD ENSEMBLE HOLDOUT TEST METRIC SCORE\n" + "="*50)
print(f"Ensemble Holdout Test R2:  {ensemble_test_metrics['R2']:.5f}")
print(f"Ensemble Holdout Test RMSE: {ensemble_test_metrics['RMSE']:.5f}")
print(f"Ensemble Holdout Test MAE:  {ensemble_test_metrics['MAE']:.5f}")

best_pred_df = test_df.copy()
best_pred_df["actual_gsi"] = y_true_test
best_pred_df["predicted_gsi"] = ensemble_test_preds
best_pred_df["residual"] = best_pred_df["actual_gsi"] - best_pred_df["predicted_gsi"]
best_pred_df["absolute_error"] = np.abs(best_pred_df["residual"])
best_pred_df.to_csv(os.path.join(CFG.OUTPUT_DIR, f"{CFG.MODEL_TO_RUN}_ensemble_test_predictions.csv"), index=False)

summary_df = pd.DataFrame([{
    "Model": CFG.MODEL_TO_RUN, "Folds": CFG.N_FOLDS,
    "OOF_Validation_R2": oof_metrics["R2"], "Ensemble_Test_R2": ensemble_test_metrics["R2"],
    "Ensemble_Test_RMSE": ensemble_test_metrics["RMSE"], "Ensemble_Test_MAE": ensemble_test_metrics["MAE"],
    "Ensemble_Test_MSE": ensemble_test_metrics["MSE"]
}])
summary_df.to_csv(os.path.join(CFG.OUTPUT_DIR, f"{CFG.MODEL_TO_RUN}_cross_validation_summary.csv"), index=False)


# ============================================================
# 15. ALIGN PREDICTIONS WITH ORIGINAL TEST TEMPLATE ORDER
# ============================================================

given_test_df = pd.read_csv(CFG.TEST_CSV)
given_test_df = clean_dataframe_columns(given_test_df)
given_test_df = clean_object_columns(given_test_df)
given_test_df["split_id"] = given_test_df[CFG.TEST_ID_COL].apply(make_id_key)

best_merge_df = best_pred_df.copy()
if "split_id" not in best_merge_df.columns:
    best_merge_df["split_id"] = best_merge_df[CFG.TEST_ID_COL].apply(make_id_key)

prediction_cols = ["split_id", "actual_gsi", "predicted_gsi", "residual", "absolute_error"]
given_test_with_predictions = given_test_df.merge(best_merge_df[prediction_cols], on="split_id", how="left")
given_test_with_predictions.to_csv(os.path.join(CFG.OUTPUT_DIR, f"{CFG.MODEL_TO_RUN}_PREDICTIONS_ON_GIVEN_TEST_GSI_CSV.csv"), index=False)


# ============================================================
# 16. UPDATED SPECIFIED GRAPHICAL PLOTS (EXACT MATCH DESIGN)
# ============================================================

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUTPUT_DIR, filename), dpi=CFG.PLOT_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    clear_memory()

# Plot 1: Target Splitting Matrix Visualization
plt.figure(figsize=(8, 5))
plt.hist(train_pool_df[TARGET_COL], bins=12, alpha=0.6, label="Total Training Pool", color="tab:blue")
plt.hist(test_df[TARGET_COL], bins=12, alpha=0.6, label="Isolated Holdout Set", color="tab:orange")
plt.xlabel("GSI Regression Value")
plt.ylabel("Data Frequency")
plt.title("Distribution of Continuous Target Matrix Across Split Pools")
plt.legend()
plt.grid(True, alpha=0.3)
save_plot(f"{CFG.MODEL_TO_RUN}_gsi_pool_distributions.png")

# Plot 2: Combined Training and Validation Loss Curves Across Folds
plt.figure(figsize=(9, 5))
for fold_idx, h_df in fold_histories.items():
    plt.plot(h_df["epoch"], h_df["train_loss"], linestyle="--", alpha=0.5, label=f"Fold {fold_idx+1} Train")
    plt.plot(h_df["epoch"], h_df["val_loss"], linestyle="-", linewidth=2, label=f"Fold {fold_idx+1} Val")
plt.xlabel("Epoch Range")
plt.ylabel("Loss Matrix Score")
plt.title(f"Objective Loss Minimization Curves: {CFG.MODEL_TO_RUN}")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
save_plot(f"{CFG.MODEL_TO_RUN}_cross_validation_loss_curves.png")

# Plot 3: Validation R2 Profile Curves Across Epoch Steps
plt.figure(figsize=(9, 5))
for fold_idx, h_df in fold_histories.items():
    plt.plot(h_df["epoch"], h_df["val_r2"], marker="o", alpha=0.7, label=f"Fold {fold_idx+1} Val R²")
plt.xlabel("Epoch")
plt.ylabel("Validation Metric Score (R²)")
plt.title(f"Validation R2 Optimization Map Across Folds: {CFG.MODEL_TO_RUN}")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
save_plot(f"{CFG.MODEL_TO_RUN}_cross_validation_r2_curves.png")


# ------------------------------------------------------------
# PLOT 4: PUBLICATION-READY ACTUAL VS PREDICTED GRAPH (MATCHED STYLE)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

# Plot structural data points matching reference styling (light blue, crisp white borders)
ax.scatter(y_true_test, ensemble_test_preds, color="#5B9BD5", edgecolors='white', s=55, alpha=0.9, zorder=3)

# Force axis constraints dynamically to produce a clean square aspect mapping ratio
min_bounds = min(y_true_test.min(), ensemble_test_preds.min()) - 5
max_bounds = max(y_true_test.max(), ensemble_test_preds.max()) + 5
ax.set_xlim(min_bounds, max_bounds)
ax.set_ylim(min_bounds, max_bounds)

# Perfect 45-degree identity reference track line
ax.plot([min_bounds, max_bounds], [min_bounds, max_bounds], linestyle="--", color="red", linewidth=1.5, zorder=2)

# Generate bounding box text matching image criteria exactly
r2_score_value = ensemble_test_metrics["R2"]
mse_score_value = ensemble_test_metrics["MSE"]

text_box_string = f"R² = {r2_score_value:.3f}"
# OPTION: If you want to explicitly append MSE into the visual text box too, uncomment below line:
# text_box_string = f"R² = {r2_score_value:.3f}\nMSE = {mse_score_value:.2f}"

props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='lightgray', alpha=1.0)
ax.text(0.08, 0.92, text_box_string, transform=ax.transAxes, fontsize=11, fontweight='bold',
        verticalalignment='top', bbox=props)

# Axis labels, alignment, and title positioning matching academic format
ax.set_xlabel("Actual GSI", fontsize=12, fontweight='bold', labelpad=12, loc='right')
ax.set_ylabel("Predicted GSI", fontsize=12, fontweight='bold', labelpad=12)
ax.set_title(f"(a) {CFG.MODEL_TO_RUN}", fontsize=13, fontweight='bold', pad=15)

# Clean up axes structure (strip top and right borders)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')

# Light dashed background grid lines
ax.grid(True, linestyle=':', alpha=0.4, color='lightgray', zorder=1)

save_plot(f"{CFG.MODEL_TO_RUN}_ensemble_actual_vs_predicted.png")
# ------------------------------------------------------------


# Plot 5: Holdout Error Range Frequency
plt.figure(figsize=(7, 5))
plt.hist(best_pred_df["residual"], bins=12, alpha=0.8, color="slategrey")
plt.xlabel("Error Dimension (True - Predicted)")
plt.ylabel("Frequency")
plt.title(f"Ensemble Holdout Error Matrix Distribution: {CFG.MODEL_TO_RUN}")
plt.grid(True, alpha=0.3)
save_plot(f"{CFG.MODEL_TO_RUN}_ensemble_residual_distribution.png")


# ============================================================
# 17. COMPRESSION EXPORT PIPELINE
# ============================================================

clear_memory()
if os.path.exists(CFG.OUTPUT_DIR + ".zip"): 
    os.remove(CFG.OUTPUT_DIR + ".zip")
shutil.make_archive(CFG.OUTPUT_DIR, "zip", CFG.OUTPUT_DIR)
print(f"\nPipeline completed. Custom CNN 5-Fold Ensemble outputs packaged at: {CFG.OUTPUT_DIR}.zip")